# WGCNA per study (v2): R notebook
### Which genes move together within each study, and which modules are biology rather than lab or cell composition?

**This is an R notebook** (R kernel; every code cell is R). It reads the files written by notebooks 2 and 3 from `rnaseq_out_v2`.

**What it does**

1. Builds a signed co-expression network **separately** for the JDM lab (JDM patients) and for our own patients.
2. Relates modules to traits **within each study**; for our patients the traits include the sequencing facility.
3. Tests JDM vs HC **inside the JDM lab**, the one contrast free of batch: module scores, gene-level limma, `mroast`.
4. Runs differential expression among our diagnoses on **raw counts** (limma-voom, batch as a term, patient as a block).
5. Compares the studies **without pooling them**.

<div class="alert alert-warning">
<b>What changed from v1, and why.</b>
<ul>
<li><b>One network per study</b> instead of one on a merged, batch-corrected matrix. On the merged matrix the two biggest modules (about 74% of genes) were a technical or compositional axis between the workbooks.</li>
<li><b>The power follows a written rule</b>: lowest power with R² of at least 0.90 where the curve has flattened, else the WGCNA FAQ fallback for the sample size. v1 defined <code>R2_TARGET = 0.80</code> but never passed it to <code>pickSoftThreshold</code>, which used its own 0.85 default.</li>
<li><b>Modules are found by their genes, not by colour.</b> Colours depend on module size and can silently change.</li>
<li><b>The <code>src_*</code> traits are removed.</b> In the merged data <code>src_goldbach</code> was the same set of samples as <code>Interferonopathy</code>, so the flag fired on every disease module.</li>
<li><b>Healthy controls from the HC workbook are not scored with patient modules</b>, because they come from a different lab.</li>
</ul>
</div>

**Roadmap:** 1 Configuration · 2 Packages · 3 Functions · 4 JDM lab · 5 Our patients · 6 Across studies · 7 How to read the results

---
# Part 1 · Configuration

🎯 **Goal:** every tunable choice in one cell.

| Setting | Meaning |
|---|---|
| `TOP_N = 4000` | genes per network (low-variance genes are mostly noise) |
| `MIN_MODULE`, `MERGE_CUT` | smallest module (30); merge modules whose eigengenes correlate above 0.75 |
| `R2_TARGET = 0.90`, `FLAT_TOL = 0.01` | the power rule |
| `POWER_FALLBACK(n)` | WGCNA FAQ for signed networks: 18 (n<20), 16 (20-30), 14 (30-40), 12 (over 40) |
| `MIN_BATCH = 4` | batches smaller than this are pooled before batch removal (a batch of one sample would erase that sample's signal) |
| `MIN_DX = 3` | smallest diagnosis group used in differential expression |
| `BATCH_FLAG_R = 0.5` | flag a module that correlates this strongly with a facility |
| `IFN_28`, `NFKB_11` | positive and negative control genes |
| `GRANULE_GENES` | neutrophil granule panel, used to find the granule module in code |

In [ ]:
options(stringsAsFactors = FALSE)

# ---- paths ------------------------------------------------------------------
MNT <- "/mnt/file-systems"
RESULTS <- "/home/jovyan/endotype_results"            # the ONE results folder
V2  <- file.path(RESULTS, "rnaseq_out_v2")
OUT <- file.path(V2, "wgcna_v2"); dir.create(OUT, showWarnings = FALSE, recursive = TRUE)
JDM_SAMPLES  <- file.path(V2, "jdm_lab_samples.csv")             # written by notebook 3
JDM_EXPR     <- file.path(V2, "jdm_lab_expression_log2QN.csv")   # written by notebook 3
OURS_SAMPLES <- file.path(V2, "rnaseq_samples.csv")              # notebook 2, one sample per patient
OURS_COUNTS  <- file.path(V2, "rnaseq_counts_raw.csv")
ELIG_SAMPLES <- file.path(V2, "rnaseq_samples_eligible.csv")     # notebook 2, all eligible samples
ELIG_COUNTS  <- file.path(V2, "rnaseq_counts_raw_eligible.csv")

# ---- network settings ---------------------------------------------------------
TOP_N <- 4000; MIN_MODULE <- 30; NETWORK <- "signed"; MERGE_CUT <- 0.25
R2_TARGET <- 0.90; FLAT_TOL <- 0.01
POWER_FALLBACK <- function(n) if (n < 20) 18 else if (n < 30) 16 else if (n < 40) 14 else 12
MIN_BATCH <- 4; MIN_DX <- 3; BATCH_FLAG_R <- 0.5

# ---- control genes and marker panels ------------------------------------------
IFN_28 <- c("CXCL10","DDX60","EPSTI1","GBP1","HERC5","HERC6","IFI27","IFI44","IFI44L","IFI6","IFIT1",
            "IFIT2","IFIT3","IFIT5","ISG15","LAMP3","LY6E","MX1","OAS1","OAS2","OAS3","OASL","RSAD2",
            "RTP4","SIGLEC1","SOCS1","SPATS2L","USP18")
NFKB_11 <- c("AICDA","CCND2","EBI3","GZMB","IFNG","MSR1","SELP","TANK","TLR2","SELL","XIAP")
GRANULE_GENES <- c("LCN2","LTF","MMP8","MPO","ELANE","AZU1","DEFA4","BPI","CEACAM8","CEACAM6",
                   "OLFM4","PRTN3","CRISP3","TCN1")
MARKERS <- list(
  erythroid_globin = c("HBA1","HBA2","HBB","HBD","HBM","ALAS2","SLC4A1","GYPA","GYPB","GYPC","CA1","AHSP","EPB42","BPGM"),
  platelet         = c("PPBP","PF4","GP9","ITGA2B","TUBB1","GNG11","SDPR"),
  T_NK             = c("CD3D","CD3E","CD2","IL7R","TRAC","LCK","CD247","NKG7","GZMA","GZMB","KLRB1"),
  B_cell           = c("CD79A","CD79B","MS4A1","CD19","IGKC","BANK1"),
  neutro_mono      = c("S100A8","S100A9","S100A12","FCGR3B","CXCR2","CSF3R","LYZ","CD14"),
  granule          = GRANULE_GENES)

show_png <- function(f, width = 900) {                  # display a saved PNG inline (used by the checkpoints)
  if (!file.exists(f)) { cat("missing:", f, "\n"); return(invisible()) }
  if (requireNamespace("IRdisplay", quietly = TRUE)) IRdisplay::display_png(file = f, width = width)
  else cat("saved to", f, "\n")
}

---
# Part 2 · Packages

🎯 **Goal:** WGCNA, limma and edgeR installed and loaded.

💡 WGCNA depends on Bioconductor packages that `install.packages()` cannot fetch, so those go through `BiocManager` first. **edgeR is new in this notebook**: it provides `DGEList`, `filterByExpr` and `calcNormFactors` for the count-based tests. If a source install fails, run `mamba install -c bioconda bioconductor-edger` in a terminal and restart the kernel. Each install runs only when the package is missing.

In [ ]:
if (!requireNamespace("BiocManager", quietly = TRUE))
  install.packages("BiocManager", repos = "https://cloud.r-project.org")
if (!requireNamespace("WGCNA", quietly = TRUE)) {
  message("installing WGCNA (first run only) ...")
  BiocManager::install(c("impute", "preprocessCore", "GO.db", "AnnotationDbi"), ask = FALSE, update = FALSE)
  install.packages("WGCNA", repos = "https://cloud.r-project.org")
}
for (p in c("limma", "edgeR"))
  if (!requireNamespace(p, quietly = TRUE)) BiocManager::install(p, ask = FALSE, update = FALSE)
suppressPackageStartupMessages({library(WGCNA); library(limma); library(edgeR)})
try(allowWGCNAThreads(4), silent = TRUE)

---
# Part 3 · Functions used throughout

🎯 **Goal:** the building blocks, defined once and used for both studies so both are treated identically.

| Function | What it does |
|---|---|
| `top_variable` | drops bad genes and samples, keeps the `TOP_N` most variable genes |
| `choose_power` | picks the soft-threshold power by the written rule and saves the plot |
| `build_network` | `blockwiseModules`, module colours, module eigengenes |
| `fit_score_model`, `score_model` | stores the SVD behind an eigengene as gene weights so **any** sample, or another study, can be scored on a module. Scoring another study uses **local standardisation** (that study's own mean and SD), as in the NIH-NLM notebooks 05b, 15b and 26b |
| `check_model` | proves the stored model reproduces WGCNA's own eigengenes (r = 1) |
| `preservation` | do a module's genes still hang together in another study? mean pairwise correlation and share of variance in PC1, each against random gene sets of the same size |
| `module_trait`, `module_summary` | module × trait correlations, each with the trait's own n, and a per-module summary |
| `pick_module` | find the interferon or granule module **by its genes** |
| `marker_table` | cell-type marker genes per module |
| `consistency_by_batch` | for a contrast, do the top genes move the same way inside every facility that holds both groups? |

In [ ]:
top_variable <- function(datExpr, n) {                         # datExpr: samples x genes
  gsg <- goodSamplesGenes(datExpr, verbose = 0)
  if (!gsg$allOK) datExpr <- datExpr[gsg$goodSamples, gsg$goodGenes, drop = FALSE]
  v <- apply(datExpr, 2, var)
  datExpr[, order(v, decreasing = TRUE)[seq_len(min(n, ncol(datExpr)))], drop = FALSE]
}

choose_power <- function(datExpr, label, outdir) {
  powers <- c(1:10, seq(12, 30, by = 2))
  invisible(capture.output(
    sft <- pickSoftThreshold(datExpr, powerVector = powers, networkType = NETWORK, verbose = 0)))
  fit <- data.frame(power = sft$fitIndices$Power,
                    scale_free_R2 = round(-sign(sft$fitIndices$slope) * sft$fitIndices$SFT.R.sq, 3),
                    mean_connectivity = round(sft$fitIndices$mean.k., 1))
  flat <- fit$power[fit$scale_free_R2 >= R2_TARGET & c(FALSE, abs(diff(fit$scale_free_R2)) < FLAT_TOL)]
  n <- nrow(datExpr)
  if (length(flat)) {
    power <- min(flat)
    rule <- sprintf("lowest power with R2 >= %.2f where the curve has flattened", R2_TARGET)
  } else {
    power <- POWER_FALLBACK(n)
    rule <- sprintf("FAQ fallback for a signed network with %d samples (no power met the rule)", n)
  }
  write.csv(fit, file.path(outdir, "soft_threshold.csv"), row.names = FALSE)
  png(file.path(outdir, "soft_threshold.png"), width = 1400, height = 600, res = 120)
  par(mfrow = c(1, 2))
  plot(fit$power, fit$scale_free_R2, type = "n", xlab = "power", ylab = "scale-free fit R^2", main = label)
  text(fit$power, fit$scale_free_R2, labels = fit$power, col = "red"); abline(h = R2_TARGET, col = "red")
  plot(fit$power, fit$mean_connectivity, type = "n", xlab = "power", ylab = "mean connectivity")
  text(fit$power, fit$mean_connectivity, labels = fit$power, col = "red")
  dev.off()
  list(power = power, rule = rule, fit = fit)
}

build_network <- function(datExpr, power) {
  assign("cor", WGCNA::cor, envir = .GlobalEnv)                # WGCNA's cor must shadow stats::cor while this runs
  on.exit(assign("cor", stats::cor, envir = .GlobalEnv), add = TRUE)
  net <- blockwiseModules(datExpr, power = power, networkType = NETWORK, TOMType = NETWORK,
                          minModuleSize = MIN_MODULE, mergeCutHeight = MERGE_CUT,
                          reassignThreshold = 0, pamRespectsDendro = FALSE,
                          numericLabels = TRUE, saveTOMs = FALSE,
                          maxBlockSize = ncol(datExpr), verbose = 0)
  colors <- labels2colors(net$colors); names(colors) <- colnames(datExpr)
  MEs <- orderMEs(moduleEigengenes(datExpr, colors = colors)$eigengenes)
  MEs <- MEs[, colnames(MEs) != "MEgrey", drop = FALSE]
  stopifnot(ncol(MEs) > 0)
  list(net = net, colors = colors, MEs = MEs)
}

plot_dendro <- function(st, file) {
  png(file, width = 1600, height = 700, res = 120)
  tryCatch(plotDendroAndColors(st$net$dendrograms[[1]], st$colors[st$net$blockGenes[[1]]],
                               "Module colors", dendroLabels = FALSE, hang = 0.03,
                               addGuide = TRUE, guideHang = 0.05),
           error = function(e) plot.new())
  dev.off()
}

In [ ]:
fit_score_model <- function(datExpr, colors) {                 # datExpr: the samples the modules were built on
  mods <- setdiff(unique(colors), "grey")
  setNames(lapply(mods, function(mo) {
    g <- names(colors)[colors == mo]
    X <- datExpr[, g, drop = FALSE]
    mu <- colMeans(X); s <- apply(X, 2, sd)
    Z <- t((t(X) - mu) / s)
    sv <- svd(t(Z), nu = 1, nv = 1)
    wt <- sv$u[, 1] / sv$d[1]
    if (cor(Z %*% wt, rowMeans(Z)) < 0) wt <- -wt              # WGCNA aligns the sign with average expression
    data.frame(module = mo, gene = g, mean = mu, sd = s, weight = wt, row.names = NULL)
  }), mods)
}

score_model <- function(E, model, local = FALSE) {             # E: genes x samples
  sapply(model, function(m) {
    m <- m[m$gene %in% rownames(E), ]
    X <- E[m$gene, , drop = FALSE]
    if (local) { mu <- rowMeans(X); s <- apply(X, 1, sd) } else { mu <- m$mean; s <- m$sd }
    keep <- is.finite(s) & s > 0
    colSums(((X[keep, , drop = FALSE] - mu[keep]) / s[keep]) * m$weight[keep])
  })
}

check_model <- function(dat, model, MEs) {                     # does the model reproduce WGCNA's eigengenes?
  sc <- score_model(t(dat), model)
  nm <- names(model)
  out <- data.frame(
    module = nm,
    r = sapply(nm, function(m) cor(sc[, m], MEs[rownames(dat), paste0("ME", m)])),
    max_abs_diff = sapply(nm, function(m) max(abs(sc[, m] - MEs[rownames(dat), paste0("ME", m)]))),
    row.names = NULL)
  if (!all(out$r > 1 - 1e-8)) warning("the scoring model does not reproduce some WGCNA eigengenes; see the table")
  out
}

preservation <- function(Zs, mod_genes, nperm = 200) {         # Zs: samples x genes, each gene standardised in THIS study
  stat <- function(Z) {
    p <- ncol(Z); d <- svd(Z, nu = 0, nv = 0)$d
    c(mean_cor = (var(rowSums(Z)) - p) / (p * (p - 1)), prop_var_PC1 = d[1]^2 / sum(d^2))
  }
  set.seed(1)
  out <- do.call(rbind, lapply(names(mod_genes), function(mo) {
    g <- intersect(mod_genes[[mo]], colnames(Zs))
    if (length(g) < 10) return(NULL)
    obs  <- stat(Zs[, g, drop = FALSE])
    null <- replicate(nperm, stat(Zs[, sample(colnames(Zs), length(g)), drop = FALSE]))
    z <- (obs - rowMeans(null)) / apply(null, 1, sd)
    data.frame(module = mo, genes_in_module = length(mod_genes[[mo]]), genes_measured = length(g),
               mean_cor = round(obs[1], 3), Z_mean_cor = round(z[1], 1),
               prop_var_PC1 = round(obs[2], 3), Z_prop_var_PC1 = round(z[2], 1), row.names = NULL)
  }))
  if (is.null(out)) return(out)
  out$reading <- as.character(cut(pmin(out$Z_mean_cor, out$Z_prop_var_PC1), c(-Inf, 2, 10, Inf),
                                  labels = c("none", "weak to moderate", "strong")))
  out[order(-pmin(out$Z_mean_cor, out$Z_prop_var_PC1)), ]
}

In [ ]:
module_trait <- function(MEs, traits) {
  traits <- traits[rownames(MEs), , drop = FALSE]
  n_per <- colSums(!is.na(traits))
  r <- cor(MEs, traits, use = "pairwise.complete.obs")
  p <- r
  for (j in seq_len(ncol(traits))) {
    n <- n_per[j]
    t <- r[, j] * sqrt((n - 2) / pmax(1 - r[, j]^2, 1e-12))
    p[, j] <- 2 * pt(-abs(t), n - 2)
  }
  list(r = r, p = p, n = n_per)
}

pick_module <- function(colors, symbols, genes) {              # the module holding the most of `genes`
  sym <- symbols[names(colors)]
  tb <- table(colors[sym %in% genes]); tb <- tb[names(tb) != "grey"]
  if (!length(tb)) return(NA_character_)
  names(tb)[which.max(tb)]
}

module_summary <- function(colors, symbols, MEs, mt, batch_cols = character(), dx_cols = character()) {
  mods <- sub("^ME", "", colnames(MEs)); sym <- symbols[names(colors)]
  rmax <- function(cols) {
    if (length(cols)) round(apply(abs(mt$r[, cols, drop = FALSE]), 1, max, na.rm = TRUE), 3)
    else rep(NA_real_, length(mods))
  }
  data.frame(module = mods, n_genes = as.integer(table(colors)[mods]),
             n_ifn_genes = unname(sapply(mods, function(m) sum(sym[colors == m] %in% IFN_28))),
             n_granule_genes = unname(sapply(mods, function(m) sum(sym[colors == m] %in% GRANULE_GENES))),
             max_abs_r_facility = unname(rmax(batch_cols)),
             max_abs_r_diagnosis = unname(rmax(dx_cols)), row.names = NULL)
}

marker_table <- function(colors, symbols) {
  sym <- symbols[names(colors)]
  sapply(MARKERS, function(mk) tapply(sym %in% mk, colors, sum))
}

plot_mt <- function(mt, file, title) {
  png(file, width = 1500, height = 300 + 55 * nrow(mt$r), res = 120)
  par(mar = c(8, 9, 3, 2))
  txt <- matrix(paste0(signif(mt$r, 2), "\n(", signif(mt$p, 1), ")"), nrow = nrow(mt$r))
  labeledHeatmap(Matrix = mt$r, xLabels = colnames(mt$r), yLabels = rownames(mt$r), ySymbols = rownames(mt$r),
                 colorLabels = FALSE, colors = blueWhiteRed(50), textMatrix = txt, setStdMargins = FALSE,
                 cex.text = 0.55, zlim = c(-1, 1), main = title)
  dev.off()
}

consistency_by_batch <- function(lc, s, A, B, tg) {            # lc: logCPM, genes x samples; columns must follow s$sample_id
  g <- tg$gene_id; sgn <- sign(tg$logFC); dx <- as.character(s$dx); bt <- as.character(s$batch)
  use <- Filter(function(b) sum(bt == b & dx == A) >= 2 && sum(bt == b & dx == B) >= 2, unique(bt))
  if (length(use) < 2 || length(g) == 0)
    return(c(usable_facilities = length(use), top_genes = length(g), same_direction_in_all = NA))
  ok <- do.call(cbind, lapply(use, function(b) {
    d <- rowMeans(lc[g, bt == b & dx == A, drop = FALSE]) - rowMeans(lc[g, bt == b & dx == B, drop = FALSE])
    sign(d) == sgn
  }))
  c(usable_facilities = length(use), top_genes = length(g), same_direction_in_all = sum(rowSums(ok) == length(use)))
}

# collapse a genes x samples matrix to one row per gene SYMBOL (keeps the most-expressed row)
to_symbol_matrix <- function(E, sym) {
  s <- sym[rownames(E)]
  ok <- !is.na(s) & s != ""
  E <- E[ok, , drop = FALSE]; s <- s[ok]
  best <- tapply(rowMeans(E), s, function(v) names(v)[which.max(v)])
  Es <- E[unlist(best), , drop = FALSE]
  rownames(Es) <- names(best)
  Es
}

# mean z-score of a gene set per sample, z-scored across the samples of THIS study
set_score <- function(Esym, genes) {
  g <- intersect(genes, rownames(Esym))
  Z <- t(scale(t(Esym[g, , drop = FALSE])))
  colMeans(Z, na.rm = TRUE)
}

summ_by <- function(score, group) {
  do.call(rbind, lapply(split(score, group), function(v)
    data.frame(n = length(v), mean = round(mean(v), 3), sd = round(sd(v), 3))))
}

---
# Part 4 · The JDM lab

🎯 **Goal:** the cleanest analysis in the project: modules built on JDM patients, then JDM vs HC **within the same lab**.

💡 **Why this order**

- **Modules are built on patients only** (the JDM samples), as the NIH-NLM notebooks do, so a disease-versus-healthy difference does not decide what the modules are. Healthy controls are then **scored** on those modules with the model fitted on the patients.
- **JDM vs HC** is tested three ways in section 4.5: module scores (Welch t-test, Benjamini-Hochberg), gene by gene (limma), and gene sets (`mroast`). All use samples processed together, so a difference cannot be a dataset effect.
- **Positive and negative controls:** the interferon response genes should be higher in JDM and the NF-kB-only genes should stay near zero. If the controls fail, nothing else should be trusted.
- **Age and sex** are unknown for most JDM-lab samples, so tests are unadjusted.

In [ ]:
# ---- load the JDM-lab study (written by notebook 3) ---------------------------
stopifnot(file.exists(JDM_SAMPLES), file.exists(JDM_EXPR))
jm <- read.csv(JDM_SAMPLES, check.names = FALSE, stringsAsFactors = FALSE)
jx <- as.matrix(read.csv(JDM_EXPR, row.names = 1, check.names = FALSE))    # genes x samples; log2 RPKM, QN within the JDM lab
jm <- jm[match(colnames(jx), jm$sample_id), ]
stopifnot(!anyNA(jm$sample_id)); rownames(jm) <- jm$sample_id
if (anyNA(jx)) warning("NA values in the JDM-lab expression matrix: ", sum(is.na(jx)))
cat("JDM lab:", ncol(jx), "samples x", nrow(jx), "genes\n")
print(table(group = jm$group))

jdm_fit <- jm$sample_id[jm$group == "JDM"]
cat("\nmodule-building samples (JDM patients only):", length(jdm_fit), "\n")
jdm_dat <- top_variable(t(jx[, jdm_fit]), TOP_N)                             # samples x genes
cat("network input:", nrow(jdm_dat), "samples x", ncol(jdm_dat), "genes\n")
sym_j <- setNames(colnames(jdm_dat), colnames(jdm_dat))                        # in this study the gene id is the symbol
OUT_J <- file.path(OUT, "jdm_lab"); dir.create(OUT_J, showWarnings = FALSE, recursive = TRUE)

### 4.1 Power, network and modules

💡 The power comes from the written rule (R² at least 0.90 where the curve has flattened, else the FAQ fallback for this sample size) and the notebook **prints which one applied**.

In [ ]:
pw_j <- choose_power(jdm_dat, "JDM lab, JDM patients", OUT_J)
cat("power", pw_j$power, "-", pw_j$rule, "\n")
st_j <- build_network(jdm_dat, pw_j$power)
cat("\nmodules found (grey = unassigned):\n")
print(sort(table(st_j$colors), decreasing = TRUE))
plot_dendro(st_j, file.path(OUT_J, "dendrogram.png"))

### 👀 Checkpoint 4.1: is the power sensible, and are the modules solid?

**Left:** R² against power with the target line; **right:** mean connectivity. If no power reached the target, the fallback was used, so treat module structure with extra caution. In the dendrogram, healthy modules are solid blocks of colour; a large grey share or heavy fragmentation means weak co-expression structure.

In [ ]:
show_png(file.path(OUT_J, "soft_threshold.png"), width = 1000)
show_png(file.path(OUT_J, "dendrogram.png"), width = 1000)

### 4.2 Scoring every sample on the modules, and proving the model

💡 A module eigengene is the first principal component of the module's genes. `fit_score_model` stores that component as gene weights so **any** sample can be scored, including healthy controls that were not used to build the modules. `check_model` must show **r = 1** against WGCNA's own eigengenes. The healthy controls are scored with the **JDM patients' mean and SD** (same lab). The interferon and granule modules are then found **by their genes**, not by colour.

In [ ]:
model_j <- fit_score_model(jdm_dat, st_j$colors)
print(check_model(jdm_dat, model_j, st_j$MEs))
scores_j <- score_model(jx, model_j)                                          # every JDM-lab sample, HC included

ifn_mod_j  <- pick_module(st_j$colors, sym_j, IFN_28)
gran_mod_j <- pick_module(st_j$colors, sym_j, GRANULE_GENES)
cat("\ninterferon module (holds the most IFN genes):", ifn_mod_j,
    "| granule module (holds the most granule genes):", gran_mod_j, "\n")
if (!is.na(ifn_mod_j) && identical(ifn_mod_j, gran_mod_j))
  warning("the interferon and granule panels landed in the same module; read the marker table (4.4) carefully")

### 4.3 Module–trait table inside the JDM patients

💡 Traits recorded for the JDM samples: age, sex, PGA (physician global assessment) and the commonest myositis-specific antibodies. **Most have missing values**, so each correlation uses that trait's own n, and only traits recorded for at least 10 samples are kept. This table says what the modules track *among patients*; it says nothing about JDM vs HC (section 4.5).

In [ ]:
jp <- jm[jdm_fit, ]
s1 <- tolower(substr(trimws(as.character(jp$sex)), 1, 1))
tr_j <- data.frame(age_yrs = suppressWarnings(as.numeric(jp$age_yrs)),
                   male = ifelse(s1 == "m", 1, ifelse(s1 == "f", 0, NA)),
                   pga = suppressWarnings(as.numeric(jp$pga)), row.names = jp$sample_id)
msa <- as.character(jp$msa); msa[is.na(msa)] <- ""
for (a in names(which(table(msa[msa != ""]) >= 5)))
  tr_j[[paste0("msa_", make.names(a))]] <- as.numeric(msa == a)
tr_j <- tr_j[, colSums(!is.na(tr_j)) >= 10, drop = FALSE]
cat("traits kept (recorded for at least 10 JDM samples):", colnames(tr_j), "\n")
if (ncol(tr_j) > 0) print(colSums(!is.na(tr_j)))

if (ncol(tr_j) > 0) {
  mt_j <- module_trait(st_j$MEs, tr_j)
  write.csv(mt_j$r, file.path(OUT_J, "module_trait_cor.csv"))
  write.csv(mt_j$p, file.path(OUT_J, "module_trait_p.csv"))
  plot_mt(mt_j, file.path(OUT_J, "module_trait_heatmap.png"), "JDM lab, JDM patients: module-trait")
} else {
  mt_j <- list(r = matrix(NA_real_, nrow(st_j$MEs), 0, dimnames = list(colnames(st_j$MEs), NULL)))
}

summ_j <- module_summary(st_j$colors, sym_j, st_j$MEs, mt_j)
summ_j$interferon_module <- summ_j$module == ifn_mod_j
summ_j$granule_module    <- summ_j$module == gran_mod_j
write.csv(summ_j, file.path(OUT_J, "module_summary.csv"), row.names = FALSE)
print(summ_j, row.names = FALSE)

### 4.4 What is each module made of?

💡 **Why this matters for whole blood:** RNA from whole blood is a mixture of cell types whose proportions differ between people and between labs. A module of haemoglobin genes is red-cell content, not disease; a module of `S100A8 / S100A9 / CSF3R` is neutrophil abundance. The table counts how many marker genes of each cell type land in each module. **A module that concentrates one column is a cell-composition module.** Marker lists are short and hand-chosen, so read the counts as a strong hint, not proof. The last lines list the leading genes of the interferon and granule modules and check that the network is built on genuinely expressed genes.

In [ ]:
mk_j <- marker_table(st_j$colors, sym_j)
print(mk_j)

kme_j <- cor(jdm_dat, st_j$MEs, use = "pairwise.complete.obs")
for (m in c(ifn_mod_j, gran_mod_j)) {
  if (is.na(m)) next
  g <- names(st_j$colors)[st_j$colors == m]
  k <- kme_j[g, paste0("ME", m)]
  cat("\n", m, "-", length(g), "genes; top 15 by module membership (kME):\n")
  print(head(names(sort(k, decreasing = TRUE)), 15))
}

mu_j <- colMeans(jdm_dat)
cat("\nmean log2(RPKM+1) of the network genes (quantiles):\n")
print(round(quantile(mu_j, c(0, .05, .25, .5, .75, 1)), 2))
cat("share of network genes with mean below 1 (RPKM below about 1):", round(mean(mu_j < 1), 3), "\n")

### 4.5 JDM vs HC inside the JDM lab

🎯 **Goal:** a batch-free test of what separates JDM from healthy controls.

💡 **Why this subset:** both groups were processed together, so a difference cannot be a dataset effect. Three views of the same contrast:

- **(a) Module scores:** each projected module score gets a Welch t-test, with Benjamini-Hochberg adjustment across modules.
- **(b) Genes:** limma on every gene, with the positive control (interferon response genes: should be **up**) and the negative control (NF-kB-only genes: should stay near **zero**).
- **(c) Gene sets:** `mroast`, which tests whether a whole pathway shifts, using rotations (a self-contained test that respects gene-gene correlation).

The tests are unadjusted for age and sex (unknown for most samples) and treat samples as independent, although some people may contribute more than one sample (see `patient_repeated_within_jdm_lab` in notebook 3).

In [ ]:
# (a) module scores, JDM vs HC
sel <- jm$sample_id[jm$group %in% c("HC", "JDM")]
grp <- factor(jm[sel, "group"], levels = c("HC", "JDM"))
cat("samples used:\n"); print(table(grp))
sc <- scores_j[sel, , drop = FALSE]
res_mod <- do.call(rbind, lapply(colnames(sc), function(m) data.frame(
  module = m,
  mean_HC  = round(mean(sc[grp == "HC",  m]), 3),
  mean_JDM = round(mean(sc[grp == "JDM", m]), 3),
  p = t.test(sc[, m] ~ grp)$p.value)))
res_mod$p_BH <- p.adjust(res_mod$p, "BH")
res_mod$interferon_module <- res_mod$module == ifn_mod_j
res_mod$granule_module    <- res_mod$module == gran_mod_j
write.csv(res_mod, file.path(OUT_J, "JDM_vs_HC_module_scores.csv"), row.names = FALSE)
cat("\nmodule scores, JDM vs HC (JDM-lab samples only):\n")
print(res_mod[order(res_mod$p), ], row.names = FALSE, digits = 3)

# (b) gene by gene
design <- model.matrix(~ grp)
fit <- eBayes(lmFit(jx[, sel], design))
tg <- topTable(fit, coef = "grpJDM", number = Inf, sort.by = "P")
write.csv(tg, file.path(OUT_J, "JDM_vs_HC_genes.csv"))
cat("\ngenes with adjusted p < 0.05:", sum(tg$adj.P.Val < 0.05), "of", nrow(tg), "\n")
ifn_p  <- intersect(IFN_28, rownames(tg)); nfk_p <- intersect(NFKB_11, rownames(tg))
cat(sprintf("positive control, interferon genes: median log2FC %.2f; %d of %d up with adjusted p < 0.05\n",
            median(tg[ifn_p, "logFC"]), sum(tg[ifn_p, "adj.P.Val"] < 0.05 & tg[ifn_p, "logFC"] > 0), length(ifn_p)))
cat(sprintf("negative control, NF-kB-only genes: median log2FC %.2f; %d of %d up with adjusted p < 0.05\n",
            median(tg[nfk_p, "logFC"]), sum(tg[nfk_p, "adj.P.Val"] < 0.05 & tg[nfk_p, "logFC"] > 0), length(nfk_p)))

### 👀 Checkpoint 4.5: eigengene scores by group

Boxplots of the four modules with the smallest p-values, JDM vs HC.

In [ ]:
top_m <- res_mod$module[order(res_mod$p)][seq_len(min(4, nrow(res_mod)))]
op <- par(mfrow = c(1, length(top_m)), mar = c(4, 4, 3, 1))
for (m in top_m)
  boxplot(sc[, m] ~ grp, main = paste0("ME", m, if (m %in% c(ifn_mod_j, gran_mod_j)) " *" else ""),
          ylab = "module score", xlab = "", col = c("#9ecae1", "#fc9272"))
par(op)
cat("* = the interferon or granule module\n")

### 4.6 Gene-set testing with `mroast` (limma)

💡 **Why `mroast`, not just per-gene tests:** a set test pools evidence across all genes of a pathway, which helps with modest sample sizes. The gene sets are the 50 Hallmark sets plus three Reactome sets (neutrophil degranulation, interferon alpha/beta signalling, interferon signalling), fetched through `msigdbr` (it handles new and old argument names and installs the `msigdbdf` data package if needed). `INCLUDE_MODULES` also tests this study's own modules as sets; that is a **sanity check, not independent evidence**, because the modules were built from the same data.

**How to read the output.** `Direction` Up = higher in JDM than HC. `PropUp` and `PropDown` are the shares of genes moving each way. `FDR` corrects for the number of sets tested. `PValue.Mixed` tests movement in either direction. A p-value of 0.0001 is the floor with 9,999 rotations, so tied sets cannot be ranked.

In [ ]:
NROT <- 9999; MIN_SET <- 10; INCLUDE_MODULES <- TRUE
set.seed(1)

get_sets <- function() {
  if (!requireNamespace("msigdbr", quietly = TRUE)) stop("msigdbr is not installed")
  try_get <- function(...) tryCatch(msigdbr::msigdbr(...), error = function(e) e)
  h <- try_get(collection = "H")
  if (inherits(h, "error") && grepl("msigdbdf", conditionMessage(h))) {
    message("installing msigdbdf (gene-set data package) ...")
    install.packages("msigdbdf", repos = c("https://igordot.r-universe.dev", "https://cloud.r-project.org"))
    h <- try_get(collection = "H")
  }
  if (inherits(h, "error")) h <- try_get(category = "H")                           # older msigdbr
  if (inherits(h, "error")) stop(conditionMessage(h))
  r <- try_get(collection = "C2", subcollection = "CP:REACTOME")
  if (inherits(r, "error")) r <- try_get(category = "C2", subcategory = "CP:REACTOME")
  if (inherits(r, "error")) stop(conditionMessage(r))
  r <- r[r$gs_name %in% c("REACTOME_NEUTROPHIL_DEGRANULATION",
                          "REACTOME_INTERFERON_ALPHA_BETA_SIGNALING",
                          "REACTOME_INTERFERON_SIGNALING"), ]
  d <- rbind(as.data.frame(h)[, c("gs_name", "gene_symbol")], as.data.frame(r)[, c("gs_name", "gene_symbol")])
  split(d$gene_symbol, d$gs_name)
}
sets <- tryCatch(get_sets(), error = function(e) { message("gene sets unavailable: ", conditionMessage(e)); NULL })

if (!is.null(sets)) {
  if (INCLUDE_MODULES) {
    ng <- st_j$colors[st_j$colors != "grey"]
    sets <- c(sets, split(names(ng), paste0("WGCNA_jdm_", ng)))
  }
  y <- jx[, sel]
  idx <- lapply(sets, function(s) which(rownames(y) %in% unique(s)))
  idx <- idx[lengths(idx) >= MIN_SET]
  cat("gene sets tested:", length(idx), "\n")
  res_ro <- mroast(y, index = idx, design = design, contrast = 2, nrot = NROT)
  res_ro <- cbind(set = rownames(res_ro), res_ro)
  write.csv(res_ro, file.path(OUT_J, "mroast_JDM_vs_HC.csv"), row.names = FALSE)
  cat("\ntop 15 sets (Direction Up = higher in JDM than HC):\n")
  print(head(res_ro[, c("set", "NGenes", "PropUp", "PropDown", "Direction", "PValue", "FDR")], 15),
        row.names = FALSE, digits = 3)
  key_sets <- c("HALLMARK_INTERFERON_ALPHA_RESPONSE", "HALLMARK_INTERFERON_GAMMA_RESPONSE",
                "REACTOME_NEUTROPHIL_DEGRANULATION", paste0("WGCNA_jdm_", c(ifn_mod_j, gran_mod_j)))
  cat("\nthe sets we care about:\n")
  print(res_ro[res_ro$set %in% key_sets, c("set", "NGenes", "Direction", "PValue", "FDR", "PValue.Mixed")],
        row.names = FALSE, digits = 3)
}

---
# Part 5 · Our patients

🎯 **Goal:** the same analysis for our own cohort, with the sequencing **facility** treated as the batch it is.

💡 **Why this is different from the JDM lab**

- **Modules are built on patients only** (AGS, CANDLE, SAVI, undifferentiated), one sample per patient. Healthy controls are left out: they come from a different workbook, so any difference between them and the patients could be the lab.
- **Normalisation follows the NIH-NLM RNA-seq notebooks:** drop genes with fewer than 10 counts in more than 90% of samples, then PFlog1pPF (scale to the mean library size, `log1p`, scale again).
- **Facility effects are removed with `removeBatchEffect`, protecting diagnosis**, and **only for building the network**. Batches with fewer than `MIN_BATCH` samples are pooled first, because a batch of one sample would be fitted exactly and erase that sample's signal. Differential expression (5.4) does *not* use this adjusted matrix; it uses raw counts with batch in the model.
- **The batch × diagnosis table is the key check.** A batch that holds a single diagnosis cannot be separated from that diagnosis.

In [ ]:
# ---- load our patients (written by notebook 2) --------------------------------
stopifnot(file.exists(OURS_SAMPLES), file.exists(OURS_COUNTS))
osm <- read.csv(OURS_SAMPLES, check.names = FALSE, stringsAsFactors = FALSE)
oc  <- read.csv(OURS_COUNTS, row.names = 1, check.names = FALSE)
gn  <- oc$Gene_Name; oc$Gene_Name <- NULL
cm  <- as.matrix(oc); storage.mode(cm) <- "numeric"
ens <- sub("\\.[0-9]+$", "", rownames(cm))                          # drop the Ensembl version
stopifnot(!anyDuplicated(ens))
rownames(cm) <- ens; names(gn) <- ens
osm <- osm[match(colnames(cm), osm$sample_id), ]
stopifnot(!anyNA(osm$sample_id)); rownames(osm) <- osm$sample_id

pat_ids <- osm$sample_id[osm$dx_clean != "HC" & osm$dx_clean != ""]
cat("one sample per patient:", ncol(cm), "samples;", length(pat_ids), "are patients (healthy controls excluded)\n\n")
cat("BATCH x DIAGNOSIS (patients):\n")
print(table(batch = osm[pat_ids, "batch"], diagnosis = osm[pat_ids, "dx_clean"]))

In [ ]:
# ---- normalise: low-count filter, then PFlog1pPF -------------------------------
keep_g <- rowMeans(cm[, pat_ids, drop = FALSE] < 10) <= 0.90
cat("genes kept after the low-count filter:", sum(keep_g), "of", length(keep_g), "\n")
pf <- function(X) t(t(X) / colSums(X) * mean(colSums(X)))
E_pf <- pf(log1p(pf(cm[keep_g, pat_ids, drop = FALSE])))              # genes x patients

# ---- remove facility effects for the network only, protecting diagnosis ---------
b_raw <- osm[pat_ids, "batch"]
tb <- table(b_raw); small <- names(tb)[tb < MIN_BATCH]
b_adj <- ifelse(b_raw %in% small, "pooled_small_batches", b_raw)
dxf <- factor(osm[pat_ids, "dx_clean"])
cat("batches:", paste(names(tb), tb, sep = "=", collapse = ", "), "\n")
if (length(small)) cat("pooled before adjustment (fewer than", MIN_BATCH, "samples):", small, "\n")
E_adj <- if (length(unique(b_adj)) > 1 && nlevels(dxf) > 1) {
  removeBatchEffect(E_pf, batch = factor(b_adj), design = model.matrix(~ dxf))
} else E_pf

OUT_O <- file.path(OUT, "ours"); dir.create(OUT_O, showWarnings = FALSE, recursive = TRUE)
png(file.path(OUT_O, "pca_before_after_batch_removal.png"), width = 1400, height = 600, res = 120)
par(mfrow = c(1, 2))
bcol <- as.integer(factor(b_adj))
for (nm in c("before", "after")) {
  M <- if (nm == "before") E_pf else E_adj
  p <- prcomp(t(M)); v <- round(100 * p$sdev[1:2]^2 / sum(p$sdev^2), 1)
  plot(p$x[, 1], p$x[, 2], col = bcol, pch = 19, xlab = paste0("PC1 (", v[1], "%)"),
       ylab = paste0("PC2 (", v[2], "%)"), main = paste("patients, coloured by facility batch:", nm))
}
legend("topright", legend = levels(factor(b_adj)), col = seq_along(levels(factor(b_adj))), pch = 19, cex = 0.6, bty = "n")
dev.off()

### 👀 Checkpoint 5.1: did removing facility effects work, without erasing diagnosis?

Patients coloured by facility batch, before and after. After removal, facilities should overlap more. Because diagnosis was protected, groups that really differ should still separate.

In [ ]:
show_png(file.path(OUT_O, "pca_before_after_batch_removal.png"), width = 1000)

### 5.2 Power, network and modules

💡 Same rule and same functions as for the JDM lab. Genes are named by Ensembl identifier (no version); `sym_o` maps them to symbols for reporting and for the cross-study step.

In [ ]:
ours_dat <- top_variable(t(E_adj), TOP_N)                              # samples x genes
sym_o <- setNames(unname(gn[colnames(ours_dat)]), colnames(ours_dat))
cat("network input:", nrow(ours_dat), "samples x", ncol(ours_dat), "genes\n")
pw_o <- choose_power(ours_dat, "our patients", OUT_O)
cat("power", pw_o$power, "-", pw_o$rule, "\n")
st_o <- build_network(ours_dat, pw_o$power)
cat("\nmodules found (grey = unassigned):\n"); print(sort(table(st_o$colors), decreasing = TRUE))
plot_dendro(st_o, file.path(OUT_O, "dendrogram.png"))

model_o <- fit_score_model(ours_dat, st_o$colors)
print(check_model(ours_dat, model_o, st_o$MEs))
ifn_mod_o  <- pick_module(st_o$colors, sym_o, IFN_28)
gran_mod_o <- pick_module(st_o$colors, sym_o, GRANULE_GENES)
cat("\ninterferon module (holds the most IFN genes):", ifn_mod_o,
    "| granule module (holds the most granule genes):", gran_mod_o, "\n")

In [ ]:
show_png(file.path(OUT_O, "soft_threshold.png"), width = 1000)
show_png(file.path(OUT_O, "dendrogram.png"), width = 1000)

### 5.3 Module–trait table, with facility traits next to diagnosis

💡 Each module eigengene is correlated with diagnosis (0/1 columns), facility batch (0/1 columns), age and sex. A module that tracks a **facility** is a possible batch module; one that tracks a **diagnosis** is a candidate for disease biology. **Read them together with the batch × diagnosis table above:** if a facility holds a single diagnosis (for example NIAID-NCI holds only undifferentiated samples), the two correlations cannot be told apart, and the flag is a warning, not a verdict.

In [ ]:
po <- osm[rownames(ours_dat), ]
tr_o <- data.frame(row.names = rownames(ours_dat))
for (d in sort(unique(po$dx_clean))) tr_o[[paste0("dx_", make.names(d))]] <- as.numeric(po$dx_clean == d)
dx_cols <- colnames(tr_o)
for (b in sort(unique(po$batch))) tr_o[[paste0("batch_", make.names(b))]] <- as.numeric(po$batch == b)
batch_cols <- setdiff(colnames(tr_o), dx_cols)
s1 <- tolower(substr(trimws(as.character(po$sex)), 1, 1))
tr_o$age_yrs <- suppressWarnings(as.numeric(po$age_yrs))
tr_o$male <- ifelse(s1 == "m", 1, ifelse(s1 == "f", 0, NA))
usable <- sapply(tr_o, function(v) sum(!is.na(v)) >= 10 && length(unique(v[!is.na(v)])) > 1)
tr_o <- tr_o[, usable, drop = FALSE]
dx_cols <- intersect(dx_cols, colnames(tr_o)); batch_cols <- intersect(batch_cols, colnames(tr_o))

mt_o <- module_trait(st_o$MEs, tr_o)
write.csv(mt_o$r, file.path(OUT_O, "module_trait_cor.csv")); write.csv(mt_o$p, file.path(OUT_O, "module_trait_p.csv"))
plot_mt(mt_o, file.path(OUT_O, "module_trait_heatmap.png"), "our patients: module-trait")

summ_o <- module_summary(st_o$colors, sym_o, st_o$MEs, mt_o, batch_cols, dx_cols)
summ_o$facility_flag   <- !is.na(summ_o$max_abs_r_facility) & summ_o$max_abs_r_facility > BATCH_FLAG_R
summ_o$interferon_module <- summ_o$module == ifn_mod_o
summ_o$granule_module    <- summ_o$module == gran_mod_o
write.csv(summ_o, file.path(OUT_O, "module_summary.csv"), row.names = FALSE)
print(summ_o, row.names = FALSE)
cat("\ncell-type markers per module:\n"); print(marker_table(st_o$colors, sym_o))

In [ ]:
show_png(file.path(OUT_O, "module_trait_heatmap.png"), width = 1000)

### 5.4 Differential expression on raw counts (limma-voom)

💡 **Why this is a separate analysis from the network:** the batch-adjusted matrix used for WGCNA is *not* used here. Adjusting first and then testing makes the p-values too optimistic. Instead, the model sees the raw counts and **facility is a term in the design**.

- **Samples:** every eligible patient sample (not only one per patient), because repeated samples from one patient are handled by a **patient block** (`duplicateCorrelation`).
- **Design:** `~ 0 + diagnosis + facility`. If diagnosis is fully nested inside facility the model cannot separate them; the code checks this (`nonEstimable`) and says so instead of returning a false answer.
- **Pairs tested:** every pair of diagnoses with at least `MIN_DX` patients each.
- **Consistency check:** the top genes must move in the same direction **inside every facility that holds both groups**. That is the check that survives a facility confound.

In [ ]:
# ---- eligible patient samples ---------------------------------------------------
es <- read.csv(ELIG_SAMPLES, check.names = FALSE, stringsAsFactors = FALSE)
ec <- read.csv(ELIG_COUNTS, row.names = 1, check.names = FALSE)
ec$Gene_Name <- NULL
em <- as.matrix(ec); storage.mode(em) <- "numeric"
rownames(em) <- sub("\\.[0-9]+$", "", rownames(em))
es <- es[match(colnames(em), es$sample_id), ]; stopifnot(!anyNA(es$sample_id))
isp <- es$dx_clean != "" & es$dx_clean != "HC"
es <- es[isp, ]; em <- em[, es$sample_id]
tbx <- table(es$batch); es$batch2 <- ifelse(es$batch %in% names(tbx)[tbx < MIN_BATCH], "pooled_small_batches", es$batch)
n_pat <- tapply(es$patient_key, es$dx_clean, function(v) length(unique(v)))
dx_ok <- names(n_pat)[n_pat >= MIN_DX]
cat("patients per diagnosis (eligible):\n"); print(n_pat)
cat("\ndiagnoses tested (>=", MIN_DX, "patients):", paste(dx_ok, collapse = ", "), "\n")
es <- es[es$dx_clean %in% dx_ok, ]; em <- em[, es$sample_id]
cat("samples used:", ncol(em), "from", length(unique(es$patient_key)), "patients\n")

In [ ]:
dge <- DGEList(counts = em)
keep <- filterByExpr(dge, group = es$dx_clean)
dge <- calcNormFactors(dge[keep, , keep.lib.sizes = FALSE])
cat("genes kept by filterByExpr:", sum(keep), "\n")
sym_all <- setNames(unname(gn[rownames(dge)]), rownames(dge))
lc_all  <- cpm(dge, log = TRUE, prior.count = 1)

dxs <- factor(es$dx_clean); bt2 <- factor(es$batch2)
des <- model.matrix(~ 0 + dxs + bt2); colnames(des) <- sub("^dxs", "", make.names(colnames(des)))
dx_names <- make.names(levels(dxs)); colnames(des)[seq_along(dx_names)] <- dx_names
ne <- nonEstimable(des)
if (!is.null(ne) && any(ne %in% dx_names)) {
  cat("!! diagnosis is not separable from facility in this design (", paste(ne, collapse = ", "),
      ").\n   Refitting WITHOUT the facility term; read every result below as possibly facility-driven.\n")
  des <- model.matrix(~ 0 + dxs); colnames(des) <- dx_names; used_batch_term <- FALSE
} else {
  if (!is.null(ne)) { des <- des[, setdiff(colnames(des), ne), drop = FALSE] }   # redundant facility columns only
  used_batch_term <- TRUE
}
cat("facility term in the model:", used_batch_term, "\n")

v <- voom(dge, des, plot = FALSE)
dup <- duplicateCorrelation(v, des, block = es$patient_key)
cat("within-patient correlation:", round(dup$consensus.correlation, 3), "\n")
v <- voom(dge, des, block = es$patient_key, correlation = dup$consensus.correlation, plot = FALSE)
fit <- lmFit(v, des, block = es$patient_key, correlation = dup$consensus.correlation)

pairs <- combn(dx_names, 2, simplify = FALSE)
de_rows <- list(); DE_DIR <- file.path(OUT_O, "de"); dir.create(DE_DIR, showWarnings = FALSE)
for (pr in pairs) {
  A <- pr[1]; B <- pr[2]
  cf <- makeContrasts(contrasts = paste0(A, " - ", B), levels = des)
  tt <- topTable(eBayes(contrasts.fit(fit, cf)), number = Inf, sort.by = "P")
  tt$gene_id <- rownames(tt); tt$symbol <- sym_all[tt$gene_id]
  write.csv(tt, file.path(DE_DIR, paste0("de_", A, "_vs_", B, ".csv")), row.names = FALSE)
  tg <- tt[tt$adj.P.Val < 0.05 & abs(tt$logFC) > 1, ]
  s_pair <- data.frame(sample_id = colnames(lc_all), dx = make.names(es$dx_clean), batch = es$batch, stringsAsFactors = FALSE)
  cons <- consistency_by_batch(lc_all, s_pair, A, B, tg)
  de_rows[[length(de_rows) + 1]] <- data.frame(
    contrast = paste(A, "vs", B), n_A = sum(make.names(es$dx_clean) == A), n_B = sum(make.names(es$dx_clean) == B),
    genes_FDR05_lfc1 = nrow(tg), up_in_A = sum(tg$logFC > 0), up_in_B = sum(tg$logFC < 0),
    ifn_genes_among_up_in_A = sum(tg$symbol[tg$logFC > 0] %in% IFN_28),
    facilities_with_both = unname(cons["usable_facilities"]),
    same_direction_in_every_facility = unname(cons["same_direction_in_all"]), row.names = NULL)
}
de_sum <- do.call(rbind, de_rows)
write.csv(de_sum, file.path(OUT_O, "de_summary.csv"), row.names = FALSE)
print(de_sum, row.names = FALSE)
cat("\nfull tables: ", DE_DIR, "\n")

### 👀 Checkpoint 5.4: how to read the differential-expression table

- **`facilities_with_both`** is how many facilities contain *both* groups with at least 2 samples each. If it is 0 or 1, the contrast is **not** protected from a facility effect and the last column is `NA`.
- **`same_direction_in_every_facility`** is how many of the significant genes move the same way in every such facility. A large share is reassuring; a small share means the list is largely facility.
- **`ifn_genes_among_up_in_A`** is a sanity check: contrasts involving CANDLE/SAVI against AGS or undifferentiated should carry interferon genes.

### 5.5 Positive and negative controls on our own data

💡 The **28-gene interferon score** should be high in the interferon-driven diagnoses; the **11-gene NF-κB panel** is a negative control that should *not* line up with them. Both are computed from log-CPM (z-score each gene across samples, average the genes), per diagnosis, in all eligible samples.

In [ ]:
Esym <- to_symbol_matrix(lc_all, sym_all)
sc_ifn  <- set_score(Esym, IFN_28)
sc_nfkb <- set_score(Esym, NFKB_11)
sc_gran <- set_score(Esym, GRANULE_GENES)
cat("genes present - IFN:", sum(IFN_28 %in% rownames(Esym)), "/", length(IFN_28),
    "| NFKB:", sum(NFKB_11 %in% rownames(Esym)), "/", length(NFKB_11),
    "| granule:", sum(GRANULE_GENES %in% rownames(Esym)), "/", length(GRANULE_GENES), "\n")
ctrl <- rbind(cbind(panel = "IFN_28", summ_by(sc_ifn, es$dx_clean), diagnosis = names(split(sc_ifn, es$dx_clean))),
              cbind(panel = "NFKB_11", summ_by(sc_nfkb, es$dx_clean), diagnosis = names(split(sc_nfkb, es$dx_clean))),
              cbind(panel = "granule", summ_by(sc_gran, es$dx_clean), diagnosis = names(split(sc_gran, es$dx_clean))))
rownames(ctrl) <- NULL; ctrl <- ctrl[, c("panel", "diagnosis", "n", "mean", "sd")]
write.csv(ctrl, file.path(OUT_O, "control_scores_by_diagnosis.csv"), row.names = FALSE)
print(ctrl, row.names = FALSE)

png(file.path(OUT_O, "control_scores.png"), width = 1500, height = 500, res = 120)
op <- par(mfrow = c(1, 3), mar = c(7, 4, 3, 1))
for (nm in c("IFN_28", "NFKB_11", "granule")) {
  sc <- switch(nm, IFN_28 = sc_ifn, NFKB_11 = sc_nfkb, granule = sc_gran)
  boxplot(sc ~ factor(es$dx_clean), las = 2, main = nm, ylab = "score", xlab = "", col = "#9ecae1")
}
par(op); dev.off()

In [ ]:
show_png(file.path(OUT_O, "control_scores.png"), width = 1100)

---
# Part 6 · Do the modules hold up in the other study?

🎯 **Goal:** test whether a module learned in one dataset exists in the other, without pooling the two.

💡 **How it works (the NIH-NLM approach)**

- Take a module's gene list from study A. In study B, standardise **each gene within B** (so lab-level offsets in mean and spread vanish), then ask whether those genes still move together.
- **Preservation statistics:** `mean_cor` (average pairwise correlation of the module's genes) and `prop_var_PC1` (share of variance on the first component), each compared with a **random-gene null** of the same size. `Z` above 10 reads strong, 2–10 weak to moderate, below 2 none.
- Genes are matched by **symbol**. Our counts are log-CPM and the JDM-lab values are log2 RPKM, but standardising within each study makes the scale difference irrelevant.
- Nothing here merges samples across labs.

In [ ]:
# symbol-keyed expression for each study (patients only, one per patient)
sym_full <- setNames(unname(gn[rownames(E_adj)]), rownames(E_adj))
Eo_sym <- to_symbol_matrix(E_adj, sym_full)                         # genes x our patients
Ej_sym <- jx[, jdm_fit]                                             # genes x JDM patients (symbols)

mods_j <- split(names(st_j$colors)[st_j$colors != "grey"], st_j$colors[st_j$colors != "grey"])
mods_o <- split(unname(sym_o[names(st_o$colors)[st_o$colors != "grey"]]),
                st_o$colors[st_o$colors != "grey"])

pz <- function(Esym) scale(t(Esym))                                  # samples x genes, each gene standardised in THIS study
pres_j_in_o <- preservation(pz(Eo_sym), mods_j)
pres_o_in_j <- preservation(pz(Ej_sym), mods_o)
if (!is.null(pres_j_in_o)) {
  pres_j_in_o$interferon_module <- pres_j_in_o$module == ifn_mod_j
  pres_j_in_o$granule_module    <- pres_j_in_o$module == gran_mod_j
}
if (!is.null(pres_o_in_j)) {
  pres_o_in_j$interferon_module <- pres_o_in_j$module == ifn_mod_o
  pres_o_in_j$granule_module    <- pres_o_in_j$module == gran_mod_o
}
write.csv(pres_j_in_o, file.path(OUT, "preservation_JDM_modules_in_ours.csv"), row.names = FALSE)
write.csv(pres_o_in_j, file.path(OUT, "preservation_our_modules_in_JDM.csv"), row.names = FALSE)
cat("JDM-lab modules tested in our patients:\n");   print(pres_j_in_o, row.names = FALSE)
cat("\nour modules tested in the JDM-lab patients:\n"); print(pres_o_in_j, row.names = FALSE)

### 👀 Checkpoint 6.1: reading the preservation tables

Look at the rows flagged `interferon_module` and `granule_module` first. A **strong** reading for the interferon module in both directions means the interferon signature is a real co-expression unit in two independent cohorts from different labs. A module that is strong in one direction only usually means the other study lacks the biology (for example, no neutrophil-rich samples) or the gene list is too small once matched by symbol (`genes_measured` against `genes_in_module`).

### 6.2 Projecting one study's modules onto the other's samples

💡 With **local standardisation** (`local = TRUE`) each gene is centred and scaled inside the target study before the stored weights are applied. Scores are comparable **within** a study, not between studies. Below, JDM-lab modules are scored on our patients by diagnosis.

In [ ]:
# JDM-lab modules (symbol-keyed model) scored on our patients, by diagnosis
sc_on_o <- score_model(Eo_sym, model_j, local = TRUE)               # our patients x JDM modules
dxo <- osm[rownames(sc_on_o), "dx_clean"]
keep_dx <- names(which(table(dxo) >= MIN_DX))
rows <- lapply(colnames(sc_on_o), function(m) {
  d <- summ_by(sc_on_o[dxo %in% keep_dx, m], dxo[dxo %in% keep_dx])
  data.frame(module = m, diagnosis = rownames(d), d, row.names = NULL)
})
proj <- do.call(rbind, rows)
proj$interferon_module <- proj$module == ifn_mod_j
proj$granule_module    <- proj$module == gran_mod_j
write.csv(proj, file.path(OUT, "jdm_modules_scored_on_our_patients.csv"), row.names = FALSE)
print(proj[proj$interferon_module | proj$granule_module, ], row.names = FALSE)

png(file.path(OUT, "jdm_ifn_granule_on_ours.png"), width = 1200, height = 500, res = 120)
op <- par(mfrow = c(1, 2), mar = c(7, 4, 3, 1))
for (m in c(ifn_mod_j, gran_mod_j)) {
  if (is.na(m)) { plot.new(); next }
  boxplot(sc_on_o[dxo %in% keep_dx, m] ~ factor(dxo[dxo %in% keep_dx]), las = 2, col = "#9ecae1",
          ylab = "score", xlab = "", main = paste0("JDM-lab module ", m,
          if (m == ifn_mod_j) " (interferon)" else " (granule)", " in our patients"))
}
par(op); dev.off()

In [ ]:
show_png(file.path(OUT, "jdm_ifn_granule_on_ours.png"), width = 1000)

### 6.3 Why there is no pooled meta-analysis here

The NIH-NLM notebooks combine per-study results (Fisher z) only after every study has been analysed **within itself**. Doing that here would need each lab to hold both patients and controls, or at least overlapping diagnoses. Ours do not:

- the JDM lab holds JDM and healthy controls only;
- our workbook holds the interferonopathies, and its healthy controls come from a different workbook, so patient-versus-control is confounded with facility.

A pooled test would therefore partly measure the lab. What this notebook can legitimately claim is **replication of structure**: the interferon module and the granule module exist in both studies (Part 6), and the one clean disease-versus-healthy contrast (JDM vs HC, Part 4) behaves as the interferon biology predicts. If a lab ever holds two overlapping diagnoses, the Part 5.4 pattern (within-study test, then combine) can be extended.

In [ ]:
# ---- environment record, for reproducibility -----------------------------------
si <- data.frame(package = c("R", "WGCNA", "limma", "edgeR", "msigdbr"),
                 version = c(paste(R.version$major, R.version$minor, sep = "."),
                             sapply(c("WGCNA", "limma", "edgeR", "msigdbr"),
                                    function(p) if (requireNamespace(p, quietly = TRUE)) as.character(packageVersion(p)) else "not installed")))
write.csv(si, file.path(OUT, "environment.csv"), row.names = FALSE)
print(si, row.names = FALSE)
cat("\nfiles written under", OUT, ":\n"); print(sort(list.files(OUT, recursive = TRUE)))

---
# Reading checklist

1. **Part 4.1 / 5.2** – did the power rule find a plateau, or was the fallback used? Fallback means weaker module structure.
2. **Part 5 batch × diagnosis table** – any facility with a single diagnosis cannot be separated from that diagnosis. Say so in any write-up.
3. **Part 4.4 / 5.3 marker tables** – is the module you care about a cell-type module (globin, platelet, neutrophil) rather than disease?
4. **Part 4.5** – JDM vs HC inside one lab: interferon control up, NF-κB control not.
5. **Part 5.4** – significant genes should keep direction inside every facility that holds both groups.
6. **Part 6** – interferon and granule modules should preserve in both directions.
7. **Not done:** no pooled analysis across labs, and healthy controls from the other workbook are never contrasted with patients.